# Funktionen für Parsen, Entitäten und Namespaces


Die Datei lb-test.xml ist eine einfache heiEditions-XML-Datei. Da werden Entites verwendet, die in https://digi.ub.uni-heidelberg.de/schema/tei/heiEDITIONS/declarations/heieditions-entities.txt" definiert sind. Im nächsten Code-Abschnitt kann man den Inhalt der Datei sehen.

In [2]:
example_file_path = '../beispiele/beispiel_data/lb-test.xml'
import codecs
with codecs.open(example_file_path, 'r', 'utf-8') as example_file:
    example = example_file.read()
    print(example)

<?xml version='1.0' encoding='UTF-8'?>
<?xml-model href="https://digi.ub.uni-heidelberg.de/schema/tei/heiEDITIONS/tei_hes.rng" type="application/xml" schematypens="http://relaxng.org/ns/structure/1.0"?><?xml-model href="https://digi.ub.uni-heidelberg.de/schema/tei/heiEDITIONS/tei_hes.rng" type="application/xml" schematypens="http://purl.oclc.org/dsdl/schematron"?>
<!DOCTYPE TEI SYSTEM "https://digi.ub.uni-heidelberg.de/schema/tei/heiEDITIONS/declarations/heieditions-entities.txt">
<TEI xmlns="http://www.tei-c.org/ns/1.0" xmlns:hei="https://digi.ub.uni-heidelberg.de/schema/tei/heiEDITIONS">
  <teiHeader>  
    <fileDesc>
      <titleStmt>
        <title ana="hc:MainTitle">Test Notes</title>
      </titleStmt>
      <publicationStmt>
        <p></p>
      </publicationStmt>
      <sourceDesc>
        <p></p>
      </sourceDesc>
    </fileDesc>
  </teiHeader>
  <facsimile>
    <surface ana="hc:Page" n="1r" xml:id="_1r">
      <zone ana="hc:VerticalFloatContainer">
        <zone ana="hc:Te

Wenn wir so eine Datei mit lxml/etree parsen wollen, kommt eine Fehlermeldung, da die externe Entities nicht geladen werden können.

In [3]:
from lxml import etree as et

try:
    tree = et.parse('../beispiele/beispiel_data/lb-test.xml')
except SyntaxError(e):
    print(e)
    pass

NameError: name 'e' is not defined

heipy bietet ein eigenes Parser, das diese Datei parsen kann.

In [ ]:
from heipy.parsers import HeiEditionsParser

heiparser = HeiEditionsParser()
tree = et.parse('../beispiele/beispiel_data/lb-test.xml', parser= heiparser)
root = tree.getroot()
print(root)

<Element {http://www.tei-c.org/ns/1.0}TEI at 0x79162616dd80>


Wenn wir in einer TEI solchen Datei XPath verwenden wollen, müssen wir entweder die Namespaces in geschweiften Klammern schreiben oder die Präfixe definieren. Also:

In [4]:
facsimile = root.findall('.//{http://www.tei-c.org/ns/1.0}facsimile')
facsimile = root.findall('.//tei:facsimile', namespaces= {'tei':'http://www.tei-c.org/ns/1.0'})

Die wichtigsten Präfixe (tei,xml,hei,hc,page,mets) werden in heipy schon in einer Variabel definiert, die importiert werden kann.

In [5]:
from heipy.namespaces import ns 

facsimile = root.findall('.//tei:facsimile', namespaces= ns)
print(facsimile)

[<Element {http://www.tei-c.org/ns/1.0}facsimile at 0x791625715640>]


Manchmal müssen wir mit lxml auch diese Präfixe vor dem Elementname schreiben. Das können wir auch mit der Funktion `prefix_format` aus heipy machen. Hier ein Beispiel, wenn wir ein neues `<link>` Element in TEI-Namespace mit etree erzeugen wollen oder alle xml:id von `<l>` Elemente suchen.

In [6]:
from heipy.namespaces import prefix_format

new_elelement = et.Element(prefix_format('tei','link'))

for line in root.findall('.//tei:l', namespaces=ns):
    line_id = line.get(prefix_format('xml','id'))
    print(line_id)

None
l_136
l_137


# Pipeline

Eine Pipeline besteht aus eine Serie von Schritten, die nacheinander ausgeführt werden. Es gibt unterschiedliche Arten von Schritten: XsltStep, PythonStep, AddAttribute, ValidationStep, DeleteStep. Diese Klassen sind in heipy.heipipe.steps definiert.

In [4]:
from heipy.heipipe.steps import Pipeline, XsltStep

pipe = Pipeline(name='Example_Pipe')

schritt1 = XsltStep(['../src/heipy/heipipe/xslt/text_initials.xsl'], name="Initials")
pipe.add_step(schritt1)

schritt2 = XsltStep(['../src/heipy/heipipe/xslt/text_markNoteAsEditorial.xsl'], name="Mark_note_as_editorial",
                    parameters= [{'note_classes': "hc:Comment"}])
pipe.add_step(schritt2)

result = pipe.execute(example_file_path)



Starting Pipeline Example_Pipe for ../beispiele/beispiel_data/lb-test.xml


Es gibt unterschiedliche Arten von Schritten: XsltStep, AddAttribute, DeleteStep, UnwrapStep, ValidationStep

In [9]:
from heipy.heipipe.steps import PythonStep, AddAttribute, ValidationStep, DeleteStep, UnwrapStep

pipe.add_step( DeleteStep(['facsimile']) )
# help(DeleteStep)

pipe.add_step(ValidationStep())
# help(ValidationStep)

pipe.add_step(AddAttribute('//tei:title', 'ana', 'hc:MainTitle'))
# help(AddAttribute)

pipe.add_step(UnwrapStep(['w']))
# help(UnwrapStep)


Help on class UnwrapStep in module heipy.heipipe.steps:

class UnwrapStep(BaseStep)
 |  UnwrapStep(elements: list, name=None, desc=None, serial=None)
 |
 |  Removes all the tags in elements, keeping the children intact
 |
 |  Method resolution order:
 |      UnwrapStep
 |      BaseStep
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(self, elements: list, name=None, desc=None, serial=None)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  __str__(self)
 |      Return str(self).
 |
 |  execute(self, input_string)
 |
 |  ----------------------------------------------------------------------
 |  Methods inherited from BaseStep:
 |
 |  add_parameter(self, param: dict)
 |
 |  get_desc(self)
 |
 |  get_name(self)
 |
 |  get_parameter_by_name(self, name: str)
 |
 |  get_parameters(self)
 |      Get the parameters for the current instance.
 |      Args:
 |          parameters (dict): A dictionary containing the parameters to be set.
 |
 |  get_se

A PythonStep is the most complex kind of step. It requires a function that takes as a parameter a root from an xml object element and a parameters argument that can be empty. For example:

In [ ]:
def add_ptr_after_l(root, parameters):
    ls = root.findall('.//tei:l', namespaces=ns)
    for l in ls:
        
